# NaturalisticDiffInt — 02: Naturalistic Pipeline

**Goal:** Run the NMPH MemoryArchive on real StudyForrest data for a single run (exploratory).

1. Download GSBS brain objects from Google Drive (GlobalStatesMemory dataset)
2. Extract BOLD timeseries (`gsbs_obj.x`) and event boundaries (`gsbs_obj.states`)
3. Load a visual feature timeseries — choose between VGG-19 (pool5) or Qwen-VL multimodal embeddings
4. PCA category/item split (Decision D6)
5. Run MemoryArchive: encode event traces, find pairmates, run competition
6. Compute RSA change matrices; save for brain comparison (notebook 03)

**Scope:** Single subject (`sub-01`) × single run (`run-1`) for rapid exploration.


## 0. GitHub Sync — Setup

In [ ]:
from google.colab import userdata
import os, subprocess

GITHUB_USER  = "drgzkr"
GITHUB_REPO  = "NaturalisticDiffInt"
REPO_PATH    = f"/content/{GITHUB_REPO}"
NOTEBOOK_REL = "notebooks/analysis/02_naturalistic_pipeline.ipynb"

_token  = userdata.get("GITHUB_TOKEN")
_remote = f"https://{_token}@github.com/{GITHUB_USER}/{GITHUB_REPO}.git"

subprocess.run(["git", "config", "--global", "user.name", "Colab"], check=True)
subprocess.run(["git", "config", "--global", "user.email", "colab@naturalistic-diffint.local"], check=True)

if not os.path.exists(REPO_PATH):
    subprocess.run(["git", "clone", _remote, REPO_PATH], check=True)
    print(f"Cloned  -> {REPO_PATH}")
else:
    subprocess.run(["git", "-C", REPO_PATH, "remote", "set-url", "origin", _remote])
    subprocess.run(["git", "-C", REPO_PATH, "pull"], check=True)
    print(f"Pulled  -> {REPO_PATH}")

del _token, _remote
print(f"Repo ready at {REPO_PATH}")


## 1. Mount Google Drive and configure paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ─── Data roots ───────────────────────────────────────────────────────────
DATA_ROOT   = "/content/project_data"          # downloaded GSBS data
QWEN_DIR    = "/content/drive/MyDrive/QWEN_Trials"   # Qwen-VL embeddings (NEMM)
VGG_DIR     = "/content/drive/MyDrive/NaturalisticDiffInt/features"  # VGG-19 (optional)
RESULTS_DIR = "/content/drive/MyDrive/NaturalisticDiffInt/results"
os.makedirs(DATA_ROOT, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print("Paths configured.")


## 2. Configuration

Set the key parameters here before running the notebook.

- **`FEATURE_MODALITY`** — `'qwen'` (Qwen-VL 2048-dim embeddings from NEMM) or `'vgg19'`
  (VGG-19 pool5 4096-dim features from GazeAware pipeline). Qwen is the primary modality for this
  exploratory run because it is multimodal (vision + language) and TR-sampled.
- **`SUB`** / **`RUN`** — subject and run for this exploration session.
- **`N_ROIS`** — number of Schaefer parcels in the GSBS objects (400 or 200).
- **`N_CATEGORY`** — PCA components treated as "category" (slow semantic) features. Top K
  slow-varying components capture scene context; remainder capture fast perceptual detail.
- **`COMPETITOR_THRESH`** — cosine similarity threshold for competitor selection in the category
  subspace. Lower values find more pairmates; start at 0.5 for Qwen (more semantic spread than VGG).


In [ ]:
import sys
sys.path.insert(0, REPO_PATH)

import math, pickle, warnings
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass, field
from typing import Optional
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from scipy import stats

warnings.filterwarnings('ignore')
torch.manual_seed(42)
np.random.seed(42)

# ─── Scope ────────────────────────────────────────────────────────────────
SUB  = 'sub-01'
RUN  = 1           # single run for exploration

# ─── Feature modality ─────────────────────────────────────────────────────
FEATURE_MODALITY = 'qwen'   # 'qwen'  |  'vgg19'

# ─── GSBS data ────────────────────────────────────────────────────────────
N_ROIS  = 400          # Schaefer parcels
TR      = 2.0          # seconds

# ─── NMPH hyperparameters (see docs/decisions.md) ─────────────────────────
N_CATEGORY        = 10    # lower for Qwen (2048-dim) — fewer components needed
COMPETITOR_THRESH = 0.50  # lower threshold for Qwen (more semantic spread)
MIN_TEMPORAL_GAP  = 3
OSC_AMP           = 0.11
OSC_PERIOD        = 10
N_OSC_TICKS       = 20
N_HIDDEN          = 64
LR                = 0.5
THETA_CROSS       = 0.5    # BCM crossover: activity > theta_cross -> integrate, else differentiate
WINDOW_SIZE       = 15    # fallback fixed-window size in TRs

print(f"Modality: {FEATURE_MODALITY}  |  Subject: {SUB}  |  Run: {RUN}")
print(f"N_CATEGORY={N_CATEGORY}  COMPETITOR_THRESH={COMPETITOR_THRESH}  OSC_AMP={OSC_AMP}")


In [ ]:
!pip install -q git+https://github.com/drgzkr/statesegmentation.git


## 3. Download GSBS brain data

The GSBS (Greedy State Boundary Search) objects are the preprocessed brain data from the
[GlobalStatesMemory project](https://github.com/drgzkr/global-states-event-memory).
Each `.npy` file contains a fitted GSBS object for one subject × run, with:

- **`.x`** — ROI × time BOLD timeseries (shape: `n_rois × n_trs`), z-scored, Schaefer parcels
- **`.states`** — integer state label per TR (shape: `n_trs`); TR boundaries where state changes = event boundaries
- **`.deltas`** — boundary strength per TR (float); higher = more likely a true event boundary
- **`.state_patterns`** — mean BOLD pattern per state (`n_states × n_rois`)
- **`.nstates`** — total number of GSBS-inferred states

These objects take ~2–3 min to download (167 MB zip, 120 GSBS files).


In [ ]:
import zipfile

GSBS_ZIP_ID   = "10Mms7DyQ87mq-jS8W7KfscGmc5QZhdmk"   # StudyForrest_SingleSubGlobalGSBS_Results.zip
GSBS_ZIP_PATH = os.path.join(DATA_ROOT, "StudyForrest_SingleSubGlobalGSBS_Results.zip")
GSBS_DIR      = os.path.join(DATA_ROOT, "StudyForrest_SingleSubGlobalGSBS_Results")

if not os.path.exists(GSBS_DIR):
    if not os.path.exists(GSBS_ZIP_PATH):
        print("Downloading GSBS objects (~167 MB)...")
        import subprocess
        subprocess.run(["pip", "install", "-q", "gdown"], check=True)
        import gdown
        gdown.download(id=GSBS_ZIP_ID, output=GSBS_ZIP_PATH, quiet=False)
    print("Unzipping...")
    with zipfile.ZipFile(GSBS_ZIP_PATH, 'r') as zf:
        zf.extractall(DATA_ROOT)
    print(f"Done -> {GSBS_DIR}")
else:
    print(f"GSBS data already present at {GSBS_DIR}")

# Quick sanity check
sample_path = os.path.join(GSBS_DIR, f"GSBS_{SUB}_run{RUN}Schaefer_{N_ROIS}_ROIs.npy")
print(f"Sample file exists: {os.path.exists(sample_path)}  — {sample_path}")


## 4. Load GSBS object and extract brain data

In [ ]:
def load_gsbs(sub, run, n_rois=400):
    path = os.path.join(GSBS_DIR, f"GSBS_{sub}_run{run}Schaefer_{n_rois}_ROIs.npy")
    obj  = np.load(path, allow_pickle=True).item()
    return obj

gsbs = load_gsbs(SUB, RUN, N_ROIS)

# ─── BOLD timeseries ──────────────────────────────────────────────────────
# gsbs.x : shape (n_rois, n_trs)  — z-scored Schaefer-400 BOLD
bold = gsbs.x                      # (n_rois, n_trs)
n_trs = bold.shape[1]

# ─── Event boundaries from GSBS states ───────────────────────────────────
states   = gsbs.states             # (n_trs,) int, state label per TR
deltas   = gsbs.deltas             # (n_trs,) float, boundary strength
nstates  = gsbs.nstates
state_patterns = gsbs.state_patterns  # (n_states, n_rois)

# Derive event onsets and durations from state labels
# A new event starts wherever states[t] != states[t-1]
state_changes = np.concatenate([[0], np.where(np.diff(states) != 0)[0] + 1])
event_onsets  = state_changes.astype(int)
event_durations = np.diff(np.append(event_onsets, n_trs)).astype(int)

print(f"Subject: {SUB}  |  Run: {RUN}")
print(f"BOLD shape  : {bold.shape}  (n_rois × n_trs)")
print(f"N TRs       : {n_trs}  ({n_trs * TR / 60:.1f} min)")
print(f"GSBS states : {nstates}")
print(f"Event onsets: {len(event_onsets)}")
print(f"State patterns: {state_patterns.shape}")


**What the GSBS object gives us:**

- The **BOLD timeseries** (`gsbs.x`) is already preprocessed: z-scored, parcellated into
  Schaefer-400 ROIs, and temporally aligned with the video. Each column is one TR (2 s).
- The **state labels** (`gsbs.states`) come from the Greedy State Boundary Search algorithm,
  which segments the BOLD signal into maximally homogeneous temporal windows. Each unique
  state label corresponds to a stable neural pattern — these are our "events" for the NMPH model.
- The **boundary strength** (`gsbs.deltas`) indicates how strongly the BOLD signal changed at each
  TR boundary. Higher values = sharper transitions = more likely perceptual/narrative event boundaries.

The event segmentation here is data-driven from the *brain* (not from visual features), which
means the NMPH model will be operating over events that are already neurally coherent. This is
more principled than fixed-window segmentation.


## 5. Load visual feature timeseries

We load one of two feature modalities, selected by `FEATURE_MODALITY` in the config:

**Qwen-VL** (`'qwen'`): 2048-dimensional multimodal embeddings from the NEMM project, extracted
from StudyForrest frames using Qwen-VL. These combine visual and linguistic scene descriptions,
making them rich semantic representations. Stored as `trial_qwenvl_embeddings_run{N}.npy`,
shape `(n_frames, 2048)`, TR-sampled.

**VGG-19** (`'vgg19'`): 4096-dimensional pool5 features from VGG-19, extracted from StudyForrest
frames. These capture lower-level visual features (textures, shapes, spatial layout). Stored as
`vgg19_pool5_run{N}.npy`, shape `(n_trs, 4096)`.

Both are already TR-sampled (one vector per 2-second TR), so they can be directly aligned with
the BOLD timeseries without resampling.


In [ ]:
def load_features(modality, run):
    if modality == 'qwen':
        path = os.path.join(QWEN_DIR, f"trial_qwenvl_embeddings_run{run}.npy")
        feats = np.load(path).astype(np.float32)
        print(f"Qwen-VL features: {feats.shape}  from {path}")
    elif modality == 'vgg19':
        path = os.path.join(VGG_DIR, f"vgg19_pool5_run{run}.npy")
        feats = np.load(path).astype(np.float32)
        print(f"VGG-19 features : {feats.shape}  from {path}")
    else:
        raise ValueError(f"Unknown modality: {modality}. Use 'qwen' or 'vgg19'.")
    return feats

feats = load_features(FEATURE_MODALITY, RUN)
n_feat_trs, feat_dim = feats.shape

print(f"Feature dim  : {feat_dim}")
print(f"Feature TRs  : {n_feat_trs}")
print(f"BOLD TRs     : {n_trs}")

# Align lengths: truncate to the shorter of the two
min_trs = min(n_trs, n_feat_trs)
if n_trs != n_feat_trs:
    print(f"\nLength mismatch — truncating both to {min_trs} TRs.")
    feats = feats[:min_trs]
    bold  = bold[:, :min_trs]
    states   = states[:min_trs]
    deltas   = deltas[:min_trs]
    # Recompute event boundaries after truncation
    state_changes = np.concatenate([[0], np.where(np.diff(states) != 0)[0] + 1])
    event_onsets  = state_changes.astype(int)
    event_durations = np.diff(np.append(event_onsets, min_trs)).astype(int)
    print(f"After truncation: {len(event_onsets)} events")

print("\nFeature loading OK.")


## 6. Event averaging and PCA category/item split

Each "event" is represented by the **mean feature vector** across all TRs within that event window.
This collapses the temporal structure within an event into a single point in feature space — analogous
to a single "stimulus presentation" in the original pairmate paradigm.

The PCA category/item split (Decision D6 in `docs/decisions.md`) decomposes the event-averaged
feature matrix into:
- **Category features** (top `N_CATEGORY` components): slow-varying, high-variance dimensions.
  These capture shared semantic/contextual structure across events — the naturalistic analogue of
  "same category" in Ritvo et al.
- **Item features** (remaining components): fast-varying, lower-variance dimensions. These
  distinguish events with similar semantic context but different perceptual content.

For Qwen features, `N_CATEGORY=10` is a reasonable starting point — the multimodal embedding
space is already semantically structured, so fewer components are needed to capture scene context.
For VGG-19 features, try `N_CATEGORY=20` (more components needed for purely visual features).


In [ ]:
def event_average(feats, onsets, durations):
    n_events = len(onsets)
    ev = np.zeros((n_events, feats.shape[1]), dtype=np.float32)
    for i, (s, d) in enumerate(zip(onsets, durations)):
        s, e = int(s), min(int(s + d), feats.shape[0])
        ev[i] = feats[s:e].mean(0)
    return ev

event_feats = event_average(feats, event_onsets, event_durations)
print(f"Event features: {event_feats.shape}  (n_events × feat_dim)")

# PCA split
scaler = StandardScaler()
X_scaled = scaler.fit_transform(event_feats)
pca = PCA(random_state=42).fit(X_scaled)

cumvar = np.cumsum(pca.explained_variance_ratio_)
n_95 = np.argmax(cumvar >= 0.95) + 1
print(f"Top {N_CATEGORY} components: {cumvar[N_CATEGORY-1]*100:.1f}% variance  |  95% at {n_95} components")

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(cumvar[:min(120, len(cumvar))], lw=2, color='steelblue')
ax.axvline(N_CATEGORY, color='red', ls='--', lw=1.5, label=f'N_CATEGORY={N_CATEGORY}')
ax.axhline(0.95, color='grey', ls=':', lw=1, label='95%')
ax.set_xlabel("PCA components")
ax.set_ylabel("Cumulative explained variance")
ax.set_title(f"{FEATURE_MODALITY.upper()} features — PCA ({SUB} run {RUN})")
ax.legend()
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()

# Project and split
proj = pca.transform(X_scaled)
cat_feats  = torch.tensor(proj[:, :N_CATEGORY], dtype=torch.float32)
item_feats = torch.tensor(proj[:, N_CATEGORY:], dtype=torch.float32)
cat_feats  = F.normalize(cat_feats, dim=1)
item_feats = F.normalize(item_feats, dim=1)

print(f"Category features: {cat_feats.shape}")
print(f"Item features    : {item_feats.shape}")


## 7. MemoryArchive (inline)

In [ ]:
@dataclass
class MemoryTrace:
    idx: int
    hidden: torch.Tensor
    hidden_init: torch.Tensor
    category_features: torch.Tensor
    item_features: torch.Tensor
    n_trs: int = 1
    onset_tr: int = 0
    competitor_idxs: list = field(default_factory=list)

    def repr_change(self):
        return F.cosine_similarity(self.hidden.unsqueeze(0), self.hidden_init.unsqueeze(0)).item()


class MemoryArchive:
    def __init__(self, n_hidden=64, competitor_threshold=0.50, min_temporal_gap=3,
                 osc_amp=0.11, osc_period=10, n_osc_ticks=20, lr=0.5, theta_cross=0.5):
        self.n_hidden            = n_hidden
        self.competitor_threshold = competitor_threshold
        self.min_temporal_gap    = min_temporal_gap
        self.osc_amp             = osc_amp
        self.osc_period          = osc_period
        self.n_osc_ticks         = n_osc_ticks
        self.lr                  = lr
        self.theta_cross         = theta_cross
        self.traces = []
        self._cp    = None
        self._ip    = None
        self.log    = []

    def _init_proj(self, nc, ni):
        self._cp = F.normalize(torch.randn(self.n_hidden, nc), dim=0)
        self._ip = F.normalize(torch.randn(self.n_hidden, ni), dim=0)

    def _project(self, cat, item):
        if self._cp is None:
            self._init_proj(cat.shape[0], item.shape[0])
        h = self._cp @ cat + self._ip @ item
        h = torch.relu(h)
        norm = h.norm()
        return h / max(norm.item(), 1e-8)

    def add_trace(self, cat, item, n_trs=1, onset_tr=0):
        h = self._project(cat, item)
        t = MemoryTrace(len(self.traces), h.clone(), h.clone(),
                        cat.clone(), item.clone(), n_trs, onset_tr)
        self.traces.append(t)
        return t

    def find_competitors(self, trace):
        out = []
        for p in self.traces:
            if p.idx >= trace.idx or (trace.idx - p.idx) < self.min_temporal_gap:
                continue
            sim = F.cosine_similarity(
                trace.category_features.unsqueeze(0),
                p.category_features.unsqueeze(0)).item()
            if sim >= self.competitor_threshold:
                out.append(p)
        return out

    def _retrieval_pass(self, tgt, comp):
        """
        Simulate oscillatory retrieval. Returns mean competitor reactivation level
        (scalar in [0,1]) — how strongly the competitor was activated during retrieval.
        """
        lv_sum = 0.0
        for t in range(self.n_osc_ticks):
            osc = 1.0 - self.osc_amp * math.sin(2 * math.pi * t / self.osc_period)
            lv  = torch.clamp(
                torch.dot(tgt.hidden, comp.hidden) / max(osc, 1e-6), 0.0, 1.0)
            lv_sum += lv.item()
        return lv_sum / self.n_osc_ticks   # scalar activity level

    def _bcm_step(self, h_target, h_comp, activity_level):
        """
        Quadratic BCM (NMPH) update.

        bcm_weight = activity * (activity - theta_cross)
            < 0  =>  differentiation (push h_target away from h_comp)
            > 0  =>  integration     (pull h_target toward h_comp)

        The magnitude scales with |activity * (activity - theta_cross)|,
        reproducing the U-shaped plasticity curve as activity sweeps [0,1].
        """
        bcm_weight = activity_level * (activity_level - self.theta_cross)
        delta      = self.lr * bcm_weight * h_comp
        updated    = torch.relu(h_target + delta)   # preserve sparsity
        norm       = updated.norm()
        return updated / max(norm.item(), 1e-8)

    def run_competition(self, trace):
        comps = self.find_competitors(trace)
        trace.competitor_idxs = [c.idx for c in comps]
        results = []
        for comp in comps:
            sb            = F.cosine_similarity(
                trace.hidden.unsqueeze(0), comp.hidden.unsqueeze(0)).item()
            activity_level = self._retrieval_pass(trace, comp)
            trace.hidden   = self._bcm_step(trace.hidden, comp.hidden, activity_level)
            sa             = F.cosine_similarity(
                trace.hidden.unsqueeze(0), comp.hidden.unsqueeze(0)).item()
            d = ('differentiation' if sa < sb - 0.005 else
                 'integration'     if sa > sb + 0.005 else 'no_change')
            e = dict(target_idx=trace.idx, competitor_idx=comp.idx,
                     sim_before=sb, sim_after=sa, delta_sim=sa - sb,
                     direction=d, competitor_activity=activity_level)
            results.append(e)
            self.log.append(e)
        return results

    def process_stream(self, cat_feats, item_feats, onsets=None):
        all_res = []
        for i in range(cat_feats.shape[0]):
            onset = int(onsets[i]) if onsets is not None else i
            trace = self.add_trace(cat_feats[i], item_feats[i], onset_tr=onset)
            all_res.extend(self.run_competition(trace))
        return all_res

    def rsa_matrix(self, use_init=False):
        if not self.traces:
            return torch.tensor([])
        vecs = torch.stack([t.hidden_init if use_init else t.hidden for t in self.traces])
        v = F.normalize(vecs, dim=1)
        return v @ v.T

    def rsa_change(self):
        return self.rsa_matrix(False) - self.rsa_matrix(True)

    def summary(self):
        dirs = [e['direction'] for e in self.log]
        ds   = [e['delta_sim'] for e in self.log]
        return dict(n_episodes=len(self.log),
                    n_diff=dirs.count('differentiation'),
                    n_intg=dirs.count('integration'),
                    n_nc=dirs.count('no_change'),
                    mean_delta=float(sum(ds)/len(ds)) if ds else 0.0)

print("MemoryArchive defined.")


## 8. Run MemoryArchive

In [ ]:
archive = MemoryArchive(
    n_hidden=N_HIDDEN,
    competitor_threshold=COMPETITOR_THRESH,
    min_temporal_gap=MIN_TEMPORAL_GAP,
    osc_amp=OSC_AMP,
    osc_period=OSC_PERIOD,
    n_osc_ticks=N_OSC_TICKS,
    lr=LR,
    theta_cross=THETA_CROSS
)

results = archive.process_stream(cat_feats, item_feats, onsets=event_onsets)
s = archive.summary()

print(f"Events encoded    : {len(archive.traces)}")
print(f"Competition episodes: {s['n_episodes']}")
print(f"Differentiation   : {s['n_diff']}  ({s['n_diff']/max(s['n_episodes'],1)*100:.1f}%)")
print(f"Integration       : {s['n_intg']}  ({s['n_intg']/max(s['n_episodes'],1)*100:.1f}%)")
print(f"No change         : {s['n_nc']}   ({s['n_nc']/max(s['n_episodes'],1)*100:.1f}%)")
print(f"Mean Δsim         : {s['mean_delta']:+.4f}")


In [ ]:
from scipy.stats import spearmanr

print("=" * 65)
print("MEMORY ARCHIVE — COMPETITION RESULTS")
print(f"  Subject: {SUB}  |  Run: {RUN}  |  Modality: {FEATURE_MODALITY.upper()}")
print("=" * 65)

s = archive.summary()
n = s['n_episodes']
pct_d = s['n_diff'] / max(n, 1) * 100
pct_i = s['n_intg'] / max(n, 1) * 100
pct_nc = s['n_nc']  / max(n, 1) * 100

print(f"  Events encoded         : {len(archive.traces)}")
print(f"  Competition episodes   : {n}")
print(f"  Differentiation        : {s['n_diff']}  ({pct_d:.1f}%)")
print(f"  Integration            : {s['n_intg']}  ({pct_i:.1f}%)")
print(f"  No change              : {s['n_nc']}   ({pct_nc:.1f}%)")
print(f"  Mean Δsim              : {s['mean_delta']:+.4f}")
print()

if not results:
    print("WARNING: NO competition episodes recorded.")
    print(f"  Current COMPETITOR_THRESH = {COMPETITOR_THRESH}")
    print("  Try lowering COMPETITOR_THRESH (e.g. to 0.3) or reducing N_CATEGORY.")
else:
    dom = 'differentiation' if s['n_diff'] > s['n_intg'] else 'integration'
    print(f"  Dominant outcome: {dom} ({max(s['n_diff'], s['n_intg'])/n*100:.1f}% of episodes)")
    all_d = [e['delta_sim'] for e in results]
    all_c = [e['competitor_activity'] for e in results]
    r_ca, p_ca = spearmanr(all_c, all_d)
    print(f"  Competitor activity × Δsim  r = {r_ca:.3f}, p = {p_ca:.4f}")
    if r_ca > 0.1 and p_ca < 0.05:
        print("  ✓  Higher competitor activity → more positive Δsim (integration) — NMPH signature.")
    elif r_ca < -0.1 and p_ca < 0.05:
        print("  ℹ  Higher competitor activity → more negative Δsim (differentiation-dominant regime).")
    print()
    sa_arr = np.array([e['sim_after'] for e in results])
    print(f"  Anticorrelated pairs (sim_after < 0): {(sa_arr < 0).sum()} ({(sa_arr < 0).mean()*100:.1f}%)")
    if (sa_arr < 0).any():
        print("    → Anticorrelated pairs are candidates for below-zero hippocampal RSA (H3).")
print("=" * 65)


**Interpreting competition outcomes:**

- **Diff%** — events driven further apart in representation space. In the brain, we predict *reduced*
  hippocampal pattern similarity between these events on re-exposure (H1 in `docs/hypotheses.md`).
- **Intg%** — events drawn closer together. Predicted to correspond to increased hippocampal RSA (H2).
- **NMPH signature** — if higher competitor activity correlates with more positive Δsim, the model is
  showing the expected U-shaped (nonmonotonic) relationship: low activity → differentiation, high
  activity → integration. This is the core prediction of the BCM learning rule applied to competing
  memory traces.

**If no episodes are found:** the semantic space is too sparse at this threshold. For Qwen features,
try lowering `COMPETITOR_THRESH` to 0.3–0.4. For VGG-19, 0.5–0.7 is typical.


## 9. BOLD event patterns (brain-side RSA)

To compare the model's representational geometry with the brain, we need to construct a
brain-side RSA matrix. We average the BOLD signal across TRs within each GSBS event window,
yielding one brain pattern per event — the same temporal alignment as the model traces.

This uses the Schaefer-400 parcellated BOLD from `gsbs.x`, which is already z-scored.
Each event's brain pattern is a 400-dimensional vector representing the mean activation
across the event window.


In [ ]:
# Average BOLD within each GSBS event window
bold_event_patterns = event_average(bold.T, event_onsets, event_durations)  # (n_events, n_rois)
print(f"BOLD event patterns: {bold_event_patterns.shape}  (n_events × n_rois)")

# Compute brain RSA matrix (cosine similarity between event patterns)
B_norm = bold_event_patterns / (np.linalg.norm(bold_event_patterns, axis=1, keepdims=True) + 1e-10)
brain_rsa = B_norm @ B_norm.T
print(f"Brain RSA matrix   : {brain_rsa.shape}")

# Model RSA change matrix
model_rsa_change = archive.rsa_change().numpy()
print(f"Model RSA change   : {model_rsa_change.shape}")

# Quick correlation between model RSA change and brain RSA (upper triangle)
n_ev = brain_rsa.shape[0]
triu_idx = np.triu_indices(n_ev, k=1)
brain_upper  = brain_rsa[triu_idx]
change_upper = model_rsa_change[triu_idx]

from scipy.stats import spearmanr as sr
r_mb, p_mb = sr(change_upper, brain_upper)
print(f"\nModel RSA change × Brain RSA: r = {r_mb:.3f}, p = {p_mb:.4f}")
if p_mb < 0.05:
    sign = 'positive' if r_mb > 0 else 'negative'
    print(f"  Significant {sign} relationship.")
    if r_mb < 0:
        print("  ✓  Negative correlation: differentiated model pairs show lower brain similarity.")
        print("     This is consistent with H1 (hippocampal RSA decrease for differentiated events).")
else:
    print("  No significant model–brain RSA correlation at this level.")
    print("  This is expected for a single run — try averaging across runs or subjects.")


## 10. Visualise RSA matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

im0 = axes[0].imshow(archive.rsa_matrix(True).numpy(), cmap='RdBu_r', vmin=-1, vmax=1)
axes[0].set_title(f"Model RSA — before competition\n({SUB} run {RUN})")
axes[0].set_xlabel("Event"); axes[0].set_ylabel("Event")
plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

im1 = axes[1].imshow(model_rsa_change, cmap='RdBu_r', vmin=-0.3, vmax=0.3)
axes[1].set_title("Model RSA change\n(blue=diff, red=intg)")
axes[1].set_xlabel("Event"); axes[1].set_ylabel("Event")
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

im2 = axes[2].imshow(brain_rsa, cmap='RdBu_r', vmin=-0.5, vmax=1.0)
axes[2].set_title(f"Brain RSA (Schaefer {N_ROIS})\n(cosine sim, event patterns)")
axes[2].set_xlabel("Event"); axes[2].set_ylabel("Event")
plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

plt.suptitle(f"{FEATURE_MODALITY.upper()} features | {SUB} run {RUN}", fontsize=12)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/rsa_matrices_{FEATURE_MODALITY}_{SUB}_run{RUN}.png", dpi=150, bbox_inches='tight')
plt.show()


**Reading the three panels:**

- **Left (Model RSA before):** Baseline representational similarity between events before NMPH
  competition. High values (red) = semantically similar events that are pairmate candidates.
  Low/negative values (blue) = semantically dissimilar events.
- **Centre (Model RSA change):** The effect of competition. Blue = differentiation (traces
  pushed apart); red = integration (traces drawn together); white = no competition occurred.
  This matrix is the key model output — its structure should predict hippocampal RSA change.
- **Right (Brain RSA):** Pairwise cosine similarity between GSBS event-averaged BOLD patterns
  (all 400 ROIs). Structured off-diagonal clusters indicate brain regions that respond similarly
  across events sharing a common narrative context.

**Key comparison:** If the model is capturing NMPH-consistent dynamics, we expect the
blue regions in the centre panel to correspond to low or negative values in the right panel
(differentiated events should also be less brain-similar). This is tested formally in notebook 03.


## 11. Pairmate scatter and competition statistics

In [ ]:
if results:
    sb  = np.array([e['sim_before'] for e in results])
    sa  = np.array([e['sim_after']  for e in results])
    ca  = np.array([e['competitor_activity'] for e in results])
    dirs = [e['direction'] for e in results]
    cmap_dict = {'differentiation': 'steelblue', 'integration': 'darkorange', 'no_change': 'grey'}
    colours = [cmap_dict[d] for d in dirs]

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    ax = axes[0]
    for d, c in cmap_dict.items():
        m = [x == d for x in dirs]
        ax.scatter(sb[m], sa[m], c=c, alpha=0.5, s=25, label=d.capitalize())
    lm = [min(sb.min(), sa.min()) - 0.05, max(sb.max(), sa.max()) + 0.05]
    ax.plot(lm, lm, 'k--', lw=0.8, alpha=0.4)
    ax.set_xlabel("Sim before"); ax.set_ylabel("Sim after")
    ax.set_title("Pairmate RSA: before vs after competition")
    ax.legend(fontsize=9)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

    ax = axes[1]
    ax.scatter(ca, sa - sb, c=colours, alpha=0.5, s=25)
    ax.axhline(0, color='k', lw=0.8, ls='--')
    ax.set_xlabel("Competitor activity (norm)")
    ax.set_ylabel("Δ sim (after − before)")
    ax.set_title("NMPH: competitor activity → direction of change")
    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(facecolor=c, label=l.capitalize()) for l, c in cmap_dict.items()], fontsize=9)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

    plt.suptitle(f"{FEATURE_MODALITY.upper()} | {SUB} run {RUN}", fontsize=11)
    plt.tight_layout()
    plt.savefig(f"{RESULTS_DIR}/pairmate_scatter_{FEATURE_MODALITY}_{SUB}_run{RUN}.png", dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("No competition episodes — try lowering COMPETITOR_THRESH.")


## 12. GSBS boundary strength and event structure

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

# Boundary strength
ax = axes[0]
time_axis = np.arange(len(deltas)) * TR
ax.plot(time_axis, deltas, color='steelblue', lw=0.8, alpha=0.7)
ax.set_ylabel("Boundary strength (δ)")
ax.set_title(f"GSBS boundary structure — {SUB} run {RUN}")
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
# Mark event onsets
for onset in event_onsets:
    ax.axvline(onset * TR, color='red', alpha=0.2, lw=0.6)

# State labels
ax = axes[1]
ax.step(time_axis, states, where='post', color='darkorange', lw=1.0)
ax.set_xlabel("Time (s)")
ax.set_ylabel("State label")
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/gsbs_structure_{SUB}_run{RUN}.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"N events   : {len(event_onsets)}")
print(f"Duration   : mean {event_durations.mean():.1f} TR  ({event_durations.mean()*TR:.1f}s)"
      f"  |  range {event_durations.min()}–{event_durations.max()} TR")
print(f"Max δ      : {deltas.max():.4f}  (strongest boundary)")
print(f"N zero-δ   : {(deltas == 0).sum()} TRs")


**Reading the GSBS boundary plot:**

- **Top panel (boundary strength δ):** Each spike indicates a neural state change detected by GSBS.
  The red vertical lines mark event onsets used in the NMPH model. High-δ boundaries correspond to
  sharp, reliable transitions — these are the strongest event boundaries.
- **Bottom panel (state labels):** The integer state label at each TR. Each flat segment is one
  GSBS-inferred neural state. This is a purely data-driven segmentation of the hippocampal signal.

**Relationship to narrative events:** GSBS state boundaries tend to align with perceptual event
boundaries (scene cuts, speaker changes) but are derived from the brain signal, not the video.
This means the NMPH model receives neurally coherent "events" rather than arbitrary windows.


## 13. Save results for notebook 03

In [ ]:
pkg = {
    'sub': SUB,
    'run': RUN,
    'modality': FEATURE_MODALITY,
    'rsa_before': archive.rsa_matrix(True).numpy(),
    'rsa_after':  archive.rsa_matrix(False).numpy(),
    'rsa_change': model_rsa_change,
    'brain_rsa':  brain_rsa,
    'log':        results,
    'n_events':   len(archive.traces),
    'event_onsets':    event_onsets,
    'event_durations': event_durations,
    'gsbs_states':   states,
    'gsbs_deltas':   deltas,
    'config': dict(
        n_category=N_CATEGORY,
        competitor_threshold=COMPETITOR_THRESH,
        min_temporal_gap=MIN_TEMPORAL_GAP,
        osc_amp=OSC_AMP,
        n_hidden=N_HIDDEN,
        n_rois=N_ROIS,
        tr=TR
    )
}

save_path = f"{RESULTS_DIR}/nmph_naturalistic_{FEATURE_MODALITY}_{SUB}_run{RUN}.pkl"
with open(save_path, 'wb') as f:
    pickle.dump(pkg, f)

print(f"Saved -> {save_path}")
print(f"Keys: {list(pkg.keys())}")


## 14. GitHub Sync — Push

In [ ]:
import json as _j, subprocess as _sp
from datetime import datetime as _dt
from google.colab import _message

_nb   = _message.blocking_request('get_ipynb', timeout_sec=30)
_dest = f"{REPO_PATH}/{NOTEBOOK_REL}"
import os as _os; _os.makedirs(_os.path.dirname(_dest), exist_ok=True)
with open(_dest, 'w') as _f: _j.dump(_nb, _f, indent=1)
_msg = f"[colab] 02_naturalistic_pipeline ({FEATURE_MODALITY}, {SUB}, run{RUN}): {_dt.now():%Y-%m-%d %H:%M}"
_sp.run(["git", "-C", REPO_PATH, "add", NOTEBOOK_REL], check=True)
_res = _sp.run(["git", "-C", REPO_PATH, "commit", "-m", _msg], capture_output=True, text=True)
if "nothing to commit" in _res.stdout:
    print("Nothing to commit.")
else:
    _sp.run(["git", "-C", REPO_PATH, "push"], check=True)
    print(f"Pushed: {_msg}")
